In [ ]:
from langchain.llms import OpenAIChat
from langchain import PromptTemplate, LLMChain
from langchain.embeddings import OpenAIEmbeddings
from langchain.callbacks import get_openai_callback
from langchain.vectorstores import Chroma
from langchain.document_loaders import TextLoader
from langchain.indexes import VectorstoreIndexCreator
from supabase import create_client

In [ ]:
import os
from os.path import abspath, join
import dotenv

In [ ]:
from llm import chat_single_query

In [ ]:
dotenv.load_dotenv('.env')

In [ ]:
chat_single_query('hi!')

In [ ]:
supabase = create_client(supabase_url=os.environ['SUPABASE_URL'], supabase_key=os.environ['SUPABASE_KEY'])

In [ ]:
dir = abspath('')

In [ ]:
name = 's41540-017-0013-4.txt'

In [ ]:
loader = TextLoader(join(dir, 'test_data', name))

In [ ]:
# with open(join(dir, 'test_data', 's41540-017-0013-4.txt'), 'r') as f:
#     full_text = f.read()

# find the abstract of an article

## save the article to supabase

In [ ]:
supabase.from_('article').insert({'name': name}).execute()

In [ ]:
query = """The following text was extracted from a PDF document.

Does this portion include part of the article's Abstract?"""

In [ ]:
query_embeddings = embeddings.embed_query(query)

In [ ]:
query_embeddings

In [ ]:
index = VectorstoreIndexCreator().from_loaders([loader])

In [ ]:
db = Chroma.from_documents(texts, document_embeddings)

# categorize

In [ ]:
options = ['chemicals', 'biochemical reactions', 'species',
           'countries', 'geological ages', 'ignition fuel',
           'planetary objects',
           'NASA projects',
           'proteins',
           'diseases', # https://disease-ontology.org
           'pharmaceutical drugs', # https://bioportal.bioontology.org/ontologies/DRON
          ]

In [ ]:
template = """The following text was extracted from a PDF document:

{text}

We are trying to categorize the document according to categories it
will discuss. The category options are:

{options}

Based on the text extract, list the categories that are likely to
exist in the full text of the article. List one category on each line.
Only provide categories from the category options list.
If no categories are expected, say 'None found'.
"""

In [ ]:
# template = """The following text was extracted from a PDF document:

# {text}

# We are trying to categorize the document according to categories it
# will discuss. The category options are:

# {options}

# Based on the text extract, list the categories that are likely to
# exist in the full text of the article. List one category on each line.
# Include any additional categories you can think of that seem relevant.
# If no categories are expected, say 'None found'.
# """

In [ ]:
llm = OpenAIChat(temperature=0)

In [ ]:
prompt = PromptTemplate(template=template, input_variables=["text", "options"])
llm_chain = LLMChain(prompt=prompt, llm=llm)
with get_openai_callback() as cb:
    # TODO how to get the right first text blob? should use embeddings
    print(llm_chain.run(text=full_text[0:3000], options=', '.join(options)))
    print(f'used {cb.total_tokens} tokens, cost ${cb.total_tokens/1000 * 0.002:.3f}')

# find stuff

In [ ]:
template = """The following text was extracted from a PDF document:

{text}

List the chemical compounds that are mentioned in the article. If you
find a chemical, provide the name of each chemical on a new line with
a description, like (chemical Name: Description). If you do not find a
chemical, say "No chemicals found".
"""
prompt = PromptTemplate(template=template, input_variables=["text"])
llm_chain = LLMChain(prompt=prompt, llm=llm)
for i in range(2):
    with get_openai_callback() as cb:
        print(llm_chain.run(full_text[2000*(i):2000*(i+1)]).replace("No chemicals found.", "").strip())
        print(cb)
        print(f'used {cb.total_tokens} tokens, cost ${cb.total_tokens/1000 * 0.002:.3f}')

In [ ]:
template = """The following text was extracted from a PDF document:

{text}

List the organisms that are being studied in the article. If you
find an organism, provide the common name and scientific name of
each organism on a new line with, like:

- Common Name (Scientific Name).

If you do not find an organism, say "No organisms found".
"""
prompt = PromptTemplate(template=template, input_variables=["text"])
llm_chain = LLMChain(prompt=prompt, llm=llm)
for t in map(''.join, zip(*[iter(full_text)]*2000)):
    print(llm_chain.run(t).replace("No organisms found.", "").replace('-', '').strip())

In [ ]:
text = f"""The following text was extracted from a PDF document:

{full_text[1000:2000]}

List the model organisms being studies that are mentioned in the article. If you
find organisms, provide each organism scientific name on a new line with no other
information. If you do not find a model organism, say "No organisms found".
"""
print(llm(text))

In [ ]:
full_text.index('SpyCas9')

In [ ]:
text = f"""The following text was extracted from a PDF document:

{full_text[29000:31000]}

List the engineered strains that are mentioned in the article. If you
find an engineered strain, provide the name of each strain on a new line with
a description, like (Strain Name: Description). If you do not find a
strain, say "No strains found".
"""
print(llm(text))

In [ ]:
paper_path = join(dir, '..', 'data', 's41586-022-05157-3.pdf')